In [1]:
!pip install -U \
  torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128
!pip install -U \
  transformers datasets accelerate evaluate scikit-learn \
  sentencepiece protobuf tiktoken \
  pandas numpy matplotlib

Looking in indexes: https://download.pytorch.org/whl/cu128
  Using cached sympy-1.14.0-py3-none-any.whl.metadata (12 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 17.2 MB/s  0:00:05 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 13.1 MB/s  0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 17.2 MB/s  0:00:00eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 15.2 MB/s  0:00:42:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 17.2 MB/s  0:00:34:00:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 17.5 MB/s  0:00:11:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.6/63.6 MB 17.8 MB/s  0:00:03 eta 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.5/267.5 MB 17.3 MB/s  0:00:15:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 288.2/288.2 MB 17.5 MB/s  0:00:16:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 17.2 MB/s  0:0

In [1]:
import os, re, csv, ast, json, random
from pathlib import Path
import numpy as np
import pandas as pd

import torch
import torch.nn as nn

from sklearn.metrics import f1_score, precision_recall_fscore_support

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

/home/kevin/venvs/pcl/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda


In [2]:
import urllib.request

DATA_TSV = Path("dontpatronizeme_pcl.tsv")
TRAIN_SPLIT_CSV = Path("train_semeval_parids-labels.csv")
DEV_SPLIT_CSV   = Path("dev_semeval_parids-labels.csv")

DATA_URL = "https://raw.githubusercontent.com/CRLala/NLPLabs-2024/main/Dont_Patronize_Me_Trainingset/dontpatronizeme_pcl.tsv"
TRAIN_SPLIT_URL = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/train_semeval_parids-labels.csv"
DEV_SPLIT_URL   = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/practice%20splits/dev_semeval_parids-labels.csv"

def download_if_missing(url: str, path: Path) -> None:
    if path.exists() and path.stat().st_size > 0:
        print(f"[OK] Found: {path} ({path.stat().st_size/1e6:.2f} MB)")
        return
    print(f"[DL] {path.name} ...")
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(path, "wb") as f:
        f.write(r.read())
    print(f"[OK] Downloaded: {path} ({path.stat().st_size/1e6:.2f} MB)")

download_if_missing(DATA_URL, DATA_TSV)
download_if_missing(TRAIN_SPLIT_URL, TRAIN_SPLIT_CSV)
download_if_missing(DEV_SPLIT_URL, DEV_SPLIT_CSV)

[OK] Found: dontpatronizeme_pcl.tsv (3.12 MB)
[OK] Found: train_semeval_parids-labels.csv (0.24 MB)
[OK] Found: dev_semeval_parids-labels.csv (0.06 MB)


In [3]:
def load_pcl_tsv(path: Path) -> pd.DataFrame:
    # Try TSV parse first
    try:
        df = pd.read_csv(
            path, sep="\t", header=None, quoting=csv.QUOTE_NONE,
            engine="python", on_bad_lines="skip"
        )
    except Exception:
        df = pd.DataFrame()

    if df.empty or df.shape[1] < 6:
        # Fallback: whitespace parsing
        rows = []
        with open(path, "r", encoding="utf-8", errors="replace") as f:
            for line in f:
                line = line.strip()
                if not line or not re.match(r"^\d+\s", line):
                    continue
                parts = line.split()
                if len(parts) < 6:
                    continue
                par_id = int(parts[0])
                art_id = parts[1].lstrip("@")
                keyword = parts[2]
                country = parts[3]
                label = int(parts[-1])
                text = " ".join(parts[4:-1])
                rows.append((par_id, art_id, keyword, country, text, label))
        return pd.DataFrame(rows, columns=["par_id","art_id","keyword","country","text","label"])

    df = df.iloc[:, :6].copy()
    df.columns = ["par_id","art_id","keyword","country","text","label"]
    df["par_id"] = df["par_id"].astype(str)
    df = df[df["par_id"].str.match(r"^\d+$")].copy()
    df["par_id"] = df["par_id"].astype(int)
    df["label"] = pd.to_numeric(df["label"], errors="coerce").astype("Int64")
    df = df.dropna(subset=["label"]).copy()
    df["label"] = df["label"].astype(int)
    df = df[df["label"].between(0,4)].copy()
    df["art_id"] = df["art_id"].astype(str).str.lstrip("@")
    df["text"] = df["text"].astype(str)
    return df

def load_split_with_categories(path: Path) -> pd.DataFrame:
    """
    Split CSV: par_id, label where label is a string "[0,1,0,0,0,0,0]"
    Returns: par_id, label_bin, cat_0..cat_6
    """
    s = pd.read_csv(path)
    s.columns = [c.strip().lower() for c in s.columns]
    if "par_id" not in s.columns or "label" not in s.columns:
        raise ValueError(f"{path} must have columns par_id,label")

    def parse_vec(x):
        vec = ast.literal_eval(x) if isinstance(x, str) else x
        vec = [int(v) for v in vec]
        assert len(vec) == 7
        return vec

    s["par_id"] = pd.to_numeric(s["par_id"], errors="coerce").astype("Int64")
    s = s.dropna(subset=["par_id","label"]).copy()
    s["par_id"] = s["par_id"].astype(int)

    cats = np.vstack(s["label"].apply(parse_vec).to_numpy())
    for i in range(7):
        s[f"cat_{i}"] = cats[:, i].astype(int)

    s["label_bin"] = (cats.sum(axis=1) > 0).astype(int)
    return s[["par_id","label_bin"] + [f"cat_{i}" for i in range(7)]]

df_all = load_pcl_tsv(DATA_TSV)
split_train = load_split_with_categories(TRAIN_SPLIT_CSV)
split_dev   = load_split_with_categories(DEV_SPLIT_CSV)

train = df_all.merge(split_train, on="par_id", how="inner")
dev   = df_all.merge(split_dev,   on="par_id", how="inner")

print("Full TSV:", df_all.shape)
print("Train:", train.shape, "| Dev:", dev.shape)
print("\nTrain label counts:")
print(train["label_bin"].value_counts())
print("\nDev label counts:")
print(dev["label_bin"].value_counts())
train.head(2)

Full TSV: (10468, 6)
Train: (8375, 14) | Dev: (2093, 14)

Train label counts:
label_bin
0    7581
1     794
Name: count, dtype: int64

Dev label counts:
label_bin
0    1894
1     199
Name: count, dtype: int64


,par_id,art_id,keyword,country,text,label,label_bin,cat_0,cat_1,cat_2,cat_3,cat_4,cat_5,cat_6
0,1,24942188,hopeless,ph,"We 're living in times of absolute insanity , ...",0,0,0,0,0,0,0,0,0
1,2,21968160,migrant,gh,"In Libya today , there are countless number of...",0,0,0,0,0,0,0,0,0


In [4]:
import html

TAG_RE = re.compile(r"<[^>]+>")

def clean_text(t: str) -> str:
    t = str(t)
    # (Required) handle HTML artifacts
    t = html.unescape(t)

    # remove any literal HTML tags if they exist
    t = TAG_RE.sub(" ", t)

    # normalize whitespace
    t = re.sub(r"\s+", " ", t).strip()
    return t

# Count “likely HTML artifacts” before cleaning (rough but useful)
def has_html_artifact(t: str) -> bool:
    t = str(t)
    return ("&" in t) or ("<" in t and ">" in t)

train["text_raw"] = train["text"]
dev["text_raw"]   = dev["text"]

train["text"] = train["text"].apply(clean_text)
dev["text"]   = dev["text"].apply(clean_text)

print("Train: suspected HTML artifacts (raw):", train["text_raw"].apply(has_html_artifact).sum())
print("Dev:   suspected HTML artifacts (raw):", dev["text_raw"].apply(has_html_artifact).sum())

# Show a couple examples where cleaning changed the string
changed = train[train["text_raw"] != train["text"]].head(5)[["text_raw","text"]]
display(changed)

Train: suspected HTML artifacts (raw): 374
Dev:   suspected HTML artifacts (raw): 95


,text_raw,text
15,"Apart from Pakistan and hosts England , Bangla...","Apart from Pakistan and hosts England , Bangla..."
37,Rizvi : There will be no joy this Eid <h> ' Th...,Rizvi : There will be no joy this Eid ' The ci...
54,"Over the past 15 years , the show has handed o...","Over the past 15 years , the show has handed o..."
55,"People who are homeless , those who were once ...","People who are homeless , those who were once ..."
68,Developing countries shoulder the most signifi...,Developing countries shoulder the most signifi...


In [33]:
MODEL_NAME = "microsoft/deberta-v3-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 160  # covers the bulk of your length distribution comfortably

ds_train = Dataset.from_pandas(train[["text","label_bin"] + [f"cat_{i}" for i in range(7)]].reset_index(drop=True))
ds_dev   = Dataset.from_pandas(dev[["text","label_bin"] + [f"cat_{i}" for i in range(7)]].reset_index(drop=True))

def tokenize_batch(batch):
    tok = tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=MAX_LEN
    )
    # IMPORTANT: Trainer expects "labels" by default for eval_loss
    tok["labels"] = batch["label_bin"]
    # Keep category labels too
    for i in range(7):
        tok[f"cat_{i}"] = batch[f"cat_{i}"]
    return tok

ds_train_tok = ds_train.map(tokenize_batch, batched=True, remove_columns=ds_train.column_names)
ds_dev_tok   = ds_dev.map(tokenize_batch, batched=True, remove_columns=ds_dev.column_names)

keep_cols = ["input_ids","attention_mask","labels"] + [f"cat_{i}" for i in range(7)]
ds_train_tok.set_format(type="torch", columns=keep_cols)
ds_dev_tok.set_format(type="torch", columns=keep_cols)

print("Sample keys:", ds_train_tok[0].keys())

Map: 100%|██████████| 2093/2093 [00:00<00:00, 27365.55 examples/s]

Sample keys: dict_keys(['cat_0', 'cat_1', 'cat_2', 'cat_3', 'cat_4', 'cat_5', 'cat_6', 'input_ids', 'attention_mask', 'labels'])


In [89]:
from transformers import AutoModel
import torch.nn as nn

class DebertaMultiTask(nn.Module):
    def __init__(self, backbone_name: str, hidden_dropout: float = 0.1):
        super().__init__()
        # FORCE FP32 weights
        self.encoder = AutoModel.from_pretrained(backbone_name, torch_dtype=torch.float32)
        hidden = self.encoder.config.hidden_size

        self.dropout = nn.Dropout(hidden_dropout)
        self.bin_head = nn.Linear(hidden, 2)   # binary logits
        self.cat_head = nn.Linear(hidden, 7)   # 7-way multi-label logits

    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls = out.last_hidden_state[:, 0, :]
        cls = self.dropout(cls)
        return self.bin_head(cls), self.cat_head(cls)

model = DebertaMultiTask(MODEL_NAME).to(device)

# sanity check
dtypes = {}
for _, p in model.named_parameters():
    dtypes[str(p.dtype)] = dtypes.get(str(p.dtype), 0) + p.numel()
print("Param dtypes:", dtypes)

Loading weights: 100%|██████████| 198/198 [00:00<00:00, 3157.58it/s, Materializing param=encoder.rel_embeddings.weight]                     
DebertaV2Model LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
mask_predictions.LayerNorm.bias         | UNEXPECTED |  | 
mask_predictions.LayerNorm.weight       | UNEXPECTED |  | 
mask_predictions.classifier.bias        | UNEXPECTED |  | 
mask_predictions.classifier.weight      | UNEXPECTED |  | 
mask_predictions.dense.bias             | UNEXPECTED |  | 
mask_predictions.dense.weight           | UNEXPECTED |  | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED |  | 
lm_predictions.lm_head.bias             | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED |  | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED |  | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be igno

Param dtypes: {'torch.float32': 183838473}


In [82]:
from transformers.trainer import Trainer

# class weights for binary CE
counts = train["label_bin"].value_counts().to_dict()
neg, pos = counts.get(0, 1), counts.get(1, 1)
# Weight positive higher
w_neg = 1.0
w_pos = neg / max(pos, 1)
class_weights = torch.tensor([w_neg, w_pos], dtype=torch.float)

print("Binary class weights:", class_weights.tolist(), "| (neg,pos)=", (neg,pos))

class MultiTaskTrainer(Trainer):
    def __init__(self, *args, alpha_aux=0.5, label_names=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.alpha_aux = alpha_aux
        self.ce = nn.CrossEntropyLoss(weight=class_weights.to(self.args.device))
        self.bce = nn.BCEWithLogitsLoss()

        # In transformers 5.2.0, label_names is not a Trainer __init__ kwarg,
        # so we store it here ourselves for the eval loop.
        if label_names is None:
            label_names = ["labels"] + [f"cat_{i}" for i in range(7)]
        self.label_names = label_names
        # Some versions internally use a private attribute
        self._label_names = label_names

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels_bin = inputs.pop("labels")
        cat = torch.stack([inputs.pop(f"cat_{i}") for i in range(7)], dim=1).float()

        bin_logits, cat_logits = model(**inputs)

        loss_bin = self.ce(bin_logits, labels_bin)
        loss_cat = self.bce(cat_logits, cat)
        loss = loss_bin + self.alpha_aux * loss_cat

        return (loss, (bin_logits, cat_logits)) if return_outputs else loss

Binary class weights: [1.0, 9.547859191894531] | (neg,pos)= (7581, 794)


In [90]:
args = TrainingArguments(
    output_dir="runs_deberta_multitask",
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    num_train_epochs=5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.06,

    logging_steps=100,

    fp16=False,
    bf16=True,
    remove_unused_columns=False,

    save_total_limit=2,
    report_to="none",
)

trainer = MultiTaskTrainer(
    model=model,
    args=args,
    train_dataset=ds_train_tok,
    eval_dataset=ds_dev_tok,
    alpha_aux=0.5,
    label_names=["labels"] + [f"cat_{i}" for i in range(7)],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [91]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.699510,0.762021
2,0.624507,0.575151
3,0.506162,0.471367
4,0.419279,0.540310
5,0.420191,0.647191


TrainOutput(global_step=2620, training_loss=0.5400576556911906, metrics={'train_runtime': 93.1577, 'train_samples_per_second': 449.507, 'train_steps_per_second': 28.124, 'total_flos': 0.0, 'train_loss': 0.5400576556911906, 'epoch': 5.0})

In [92]:
@torch.no_grad()
def get_dev_logits(trainer, ds):
    loader = torch.utils.data.DataLoader(ds, batch_size=64)
    model = trainer.model
    model.eval()

    all_logits = []
    all_y = []

    for batch in loader:
        input_ids = batch["input_ids"].to(trainer.args.device)
        attn      = batch["attention_mask"].to(trainer.args.device)

        # labels are stored under "labels" now
        y = batch["labels"].cpu().numpy()

        bin_logits, _ = model(input_ids=input_ids, attention_mask=attn)

        # positive-class logit (index 1)
        pos_logit = bin_logits[:, 1].detach().cpu().numpy()

        all_logits.append(pos_logit)
        all_y.append(y)

    return np.concatenate(all_logits), np.concatenate(all_y)

dev_logits, dev_y = get_dev_logits(trainer, ds_dev_tok)

# Threshold sweep (maximize positive-class F1)
ths = np.linspace(dev_logits.min(), dev_logits.max(), 200)
best = (-1, None, None, None)  # f1, thr, prec, rec

for thr in ths:
    pred = (dev_logits >= thr).astype(int)
    p, r, f1, _ = precision_recall_fscore_support(
        dev_y, pred, average="binary", pos_label=1, zero_division=0
    )
    if f1 > best[0]:
        best = (f1, thr, p, r)

best_f1, best_thr, best_p, best_r = best
print(f"Best dev positive-class F1: {best_f1:.4f}")
print(f"  threshold: {best_thr:.4f} | precision: {best_p:.4f} | recall: {best_r:.4f}")

# Compare with default threshold ~0 (roughly p>=0.5 if logits are well-calibrated)
default_pred = (dev_logits >= 0).astype(int)
p0, r0, f10, _ = precision_recall_fscore_support(
    dev_y, default_pred, average="binary", pos_label=1, zero_division=0
)
print(f"Default (thr=0) F1: {f10:.4f} | precision: {p0:.4f} | recall: {r0:.4f}")

Best dev positive-class F1: 0.5203
  threshold: -0.5682 | precision: 0.4519 | recall: 0.6131
Default (thr=0) F1: 0.5058 | precision: 0.4698 | recall: 0.5477


In [93]:
from pathlib import Path
import json

BEST_DIR = Path("BestModel")
BEST_DIR.mkdir(exist_ok=True)

# Save HF-style checkpoint (recommended)
trainer.save_model(BEST_DIR)          # saves model weights + config
tokenizer.save_pretrained(BEST_DIR)   # saves tokenizer files

# Save extra metadata (threshold etc.)
with open(BEST_DIR / "model_info.json", "w") as f:
    json.dump(
        {
            "backbone": MODEL_NAME,
            "max_length": MAX_LEN,
            "alpha_aux": 0.5,
            "best_threshold_dev": float(best_thr),
            "best_f1_dev": float(best_f1),
            "precision_dev": float(best_p),
            "recall_dev": float(best_r),
        },
        f, indent=2
    )

print("Saved to:", BEST_DIR.resolve())

Saved to: /home/kevin/Dev/Imperial/NLPCW/BestModel


In [104]:
import re, urllib.request
from pathlib import Path
import numpy as np
import pandas as pd
import torch

START_RE = re.compile(r"(t_\d+)\s+@@(\d+)\s+([A-Za-z0-9\-]+)\s+([A-Za-z]{2})\s+")

def read_dpm_tsv(path):
    rows = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.rstrip("\n")
            if not line or not line[0].isdigit():
                continue
            parts = line.split("\t")
            if len(parts) >= 6:
                par_id = int(parts[0])
                text = "\t".join(parts[4:-1])
            else:
                ws = line.split()
                if len(ws) < 6:
                    continue
                par_id = int(ws[0])
                text = " ".join(ws[4:-1])
            rows.append((par_id, clean_text(text)))
    return pd.DataFrame(rows, columns=["par_id", "text"]).drop_duplicates("par_id")

def fetch_if_missing(url, path):
    path = Path(path)
    if path.exists() and path.stat().st_size > 0:
        return path
    req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
    with urllib.request.urlopen(req) as r, open(path, "wb") as f:
        f.write(r.read())
    return path

def read_task4_test(path):
    raw = Path(path).read_text(encoding="utf-8", errors="replace")
    flat = re.sub(r"\s+", " ", raw).strip()
    ms = list(START_RE.finditer(flat))
    if not ms:
        raise ValueError("couldn't parse task4_test.tsv")
    texts = []
    for i, m in enumerate(ms):
        s = m.end()
        e = ms[i+1].start() if i+1 < len(ms) else len(flat)
        texts.append(clean_text(flat[s:e].strip().strip('"')))
    return texts

@torch.no_grad()
def predict(texts, batch_size=64):
    trainer.model.eval()
    out = []
    for i in range(0, len(texts), batch_size):
        tok = tokenizer(
            texts[i:i+batch_size],
            truncation=True,
            padding="max_length",
            max_length=MAX_LEN,
            return_tensors="pt",
        )
        tok = {k: v.to(trainer.args.device) for k, v in tok.items() if k in ["input_ids", "attention_mask"]}
        logits, _ = trainer.model(**tok)
        pos = logits[:, 1].detach().cpu().numpy()
        out.append((pos >= best_thr).astype(int))
    return np.concatenate(out)

# dev.txt (official dev order)
full_df = read_dpm_tsv(DATA_TSV)
dev_texts = split_dev[["par_id"]].merge(full_df, on="par_id", how="left")["text"].tolist()
if any(t is None or (isinstance(t, float) and np.isnan(t)) for t in dev_texts):
    missing = split_dev.loc[pd.isna(split_dev[["par_id"]].merge(full_df, on="par_id", how="left")["text"]), "par_id"].head(20).tolist()
    raise ValueError(f"missing dev texts for par_id(s) like {missing}")

dev_pred = predict(dev_texts)
Path("dev.txt").write_text("\n".join(map(str, dev_pred.tolist())) + "\n", encoding="utf-8")
print("dev.txt lines:", len(dev_pred))

# test.txt (don’t redownload if already present)
TEST_URL = "https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/TEST/task4_test.tsv"
test_path = fetch_if_missing(TEST_URL, "task4_test.tsv")
test_texts = read_task4_test(test_path)

test_pred = predict(test_texts)
Path("test.txt").write_text("\n".join(map(str, test_pred.tolist())) + "\n", encoding="utf-8")
print("test.txt lines:", len(test_pred))

dev.txt lines: 2094
test.txt lines: 3832


In [105]:
# Check dev.txt vs dev_semeval_parids-labels.csv (positive-class F1)
import numpy as np
import pandas as pd
import ast
from sklearn.metrics import precision_recall_fscore_support

DEV_PRED_PATH = "dev.txt"
DEV_SPLIT_CSV = "dev_semeval_parids-labels.csv"  # adjust if needed

# 1) Load predictions (one 0/1 per line)
pred = []
with open(DEV_PRED_PATH, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line == "":
            continue
        pred.append(int(line))
pred = np.array(pred, dtype=int)

# 2) Load gold labels from split file (7-dim vector -> binary = any bit)
gold_df = pd.read_csv(DEV_SPLIT_CSV)  # columns: par_id,label
def vec_to_bin(x):
    v = ast.literal_eval(x) if isinstance(x, str) else x
    return int(any(int(t) == 1 for t in v))
gold = gold_df["label"].apply(vec_to_bin).to_numpy(dtype=int)

# 3) Sanity checks on length
print("Pred lines:", len(pred))
print("Gold lines:", len(gold))
assert len(pred) == len(gold), "Length mismatch: dev.txt must have exactly one line per dev example (in the same order)."

# 4) Positive-class metrics (PCL=1)
p, r, f1, _ = precision_recall_fscore_support(gold, pred, average="binary", pos_label=1, zero_division=0)
acc = (pred == gold).mean()

print(f"Dev Accuracy: {acc:.4f}")
print(f"Dev Positive-class Precision: {p:.4f}")
print(f"Dev Positive-class Recall:    {r:.4f}")
print(f"Dev Positive-class F1:        {f1:.4f}")

# (Optional) Confusion counts
tp = int(((pred == 1) & (gold == 1)).sum())
fp = int(((pred == 1) & (gold == 0)).sum())
fn = int(((pred == 0) & (gold == 1)).sum())
tn = int(((pred == 0) & (gold == 0)).sum())
print(f"Confusion: TP={tp} FP={fp} FN={fn} TN={tn}")

Pred lines: 2094
Gold lines: 2094
Dev Accuracy: 0.8926
Dev Positive-class Precision: 0.4519
Dev Positive-class Recall:    0.6131
Dev Positive-class F1:        0.5203
Confusion: TP=122 FP=148 FN=77 TN=1747
